<a href="https://colab.research.google.com/github/natyherreraa/Pre-Entrega-6/blob/main/Pre_Entrega_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-entrega 6 · Análisis de temperaturas en Aeroparque y Ezeiza

En este trabajo se analiza cómo variaron las temperaturas en dos aeropuertos, Aeroparque Jorge Newbery y Ezeiza Ministro Pistarini entre enero y agosto de 2025. Se utilizan registros meteorológicos diarios del conjunto *Global Surface Summary of the Day* (GSOD) de la NOAA.

El análisis se realiza con Python y DuckDB. Mediante una conexión en memoria, se consultan los archivos CSV directamente con SQL.

Primero se verifica si hay temperaturas faltantes o fechas duplicadas por aeropuerto y se convierten las temperaturas de Fahrenheit a Celsius.

Luego se calculan los promedios mensuales y las máximas del período, se cuentan los días cálidos y se identifican las fechas con mayor diferencia de temperatura media entre ambos aeropuertos.



In [1]:
# Importar las librerias y herramientas
import duckdb
import pandas as pd

# Abrir una conexión en memoria
conn = duckdb.connect(":memory:")

In [2]:
# Conexion al dataset y creación de vista


archivos_csv = [
    "https://noaa-gsod-pds.s3.amazonaws.com/2025/87582099999.csv",
    "https://noaa-gsod-pds.s3.amazonaws.com/2025/87576099999.csv"
]

conn.read_csv(archivos_csv, header=True).create_view("clima", replace=True)

# Contar cuántos registros hay por aeropuerto
conn.execute("""
    SELECT NAME AS aeropuerto, COUNT(*) AS registros
    FROM clima
    GROUP BY NAME
    ORDER BY aeropuerto
""").df()

,aeropuerto,registros
0,"AEROPARQUE JORGE NEWBERY, AR",236
1,"MINISTRO PISTARINI, AR",236


In [3]:
# Contar faltantes en las temperaturas de cada aeropuerto

conn.execute("""
    SELECT
        NAME AS aeropuerto,
        COUNT(*) AS registros,
        COUNT(*) FILTER (WHERE TEMP IS NULL OR TEMP = 9999.9) AS medias_faltantes,
        COUNT(*) FILTER (WHERE "MAX" IS NULL OR "MAX" = 9999.9) AS maximas_faltantes,
        COUNT(*) FILTER (WHERE "MIN" IS NULL OR "MIN" = 9999.9) AS minimas_faltantes
    FROM clima
    GROUP BY NAME
    ORDER BY aeropuerto
""").df()

,aeropuerto,registros,medias_faltantes,maximas_faltantes,minimas_faltantes
0,"AEROPARQUE JORGE NEWBERY, AR",236,0,0,0
1,"MINISTRO PISTARINI, AR",236,0,0,0


In [4]:
# Verificar si algún aeropuerto tiene más de un registro para la misma fecha


# Buscar días con más de un registro para el mismo aeropuerto
conn.execute("""
    SELECT
        NAME AS aeropuerto,
        CAST("DATE" AS DATE) AS fecha,
        COUNT(*) AS cantidad_registros
    FROM clima
    GROUP BY NAME, CAST("DATE" AS DATE)
    HAVING COUNT(*) > 1
    ORDER BY aeropuerto, fecha
""").df()

,aeropuerto,fecha,cantidad_registros


In [5]:
# Preparar los datos para el análisis y actualizar la vista clima
datos = conn.read_csv(archivos_csv, header=True)

datos.project("""
    NAME AS aeropuerto,
    CAST("DATE" AS DATE) AS fecha,
    (NULLIF(TRY_CAST(TEMP AS DOUBLE), 9999.9) - 32) * 5.0 / 9 AS media_c,
    (NULLIF(TRY_CAST("MAX" AS DOUBLE), 9999.9) - 32) * 5.0 / 9 AS maxima_c,
    (NULLIF(TRY_CAST("MIN" AS DOUBLE), 9999.9) - 32) * 5.0 / 9 AS minima_c
""").create_view("clima", replace=True)

# Mostrar cinco registros con las temperaturas en grados Celsius
conn.execute("""
    SELECT aeropuerto, fecha, media_c, maxima_c, minima_c
    FROM clima
    ORDER BY fecha, aeropuerto
    LIMIT 5
""").df()

,aeropuerto,fecha,media_c,maxima_c,minima_c
0,"AEROPARQUE JORGE NEWBERY, AR",2025-01-01,25.444444,29.111111,22.777778
1,"MINISTRO PISTARINI, AR",2025-01-01,27.888889,37.000000,21.500000
2,"AEROPARQUE JORGE NEWBERY, AR",2025-01-02,26.333333,31.500000,20.000000
3,"MINISTRO PISTARINI, AR",2025-01-02,23.888889,36.000000,15.222222
4,"AEROPARQUE JORGE NEWBERY, AR",2025-01-03,22.944444,31.611111,17.222222


In [6]:
# Calcular la temperatura media mensual por aeropuerto

conn.execute("""
    SELECT
        aeropuerto,
        DATE_TRUNC('month', fecha)::DATE AS mes,
        COUNT(media_c) AS dias_con_dato,
        ROUND(AVG(media_c), 2) AS temperatura_media_c
    FROM clima
    GROUP BY aeropuerto, mes
    ORDER BY mes, aeropuerto
""").df()

,aeropuerto,mes,dias_con_dato,temperatura_media_c
0,"AEROPARQUE JORGE NEWBERY, AR",2025-01-01,31,24.91
1,"MINISTRO PISTARINI, AR",2025-01-01,31,25.38
2,"AEROPARQUE JORGE NEWBERY, AR",2025-02-01,28,25.12
3,"MINISTRO PISTARINI, AR",2025-02-01,28,25.00
4,"AEROPARQUE JORGE NEWBERY, AR",2025-03-01,31,22.61
5,"MINISTRO PISTARINI, AR",2025-03-01,31,21.97
6,"AEROPARQUE JORGE NEWBERY, AR",2025-04-01,30,17.89
7,"MINISTRO PISTARINI, AR",2025-04-01,30,16.37
8,"AEROPARQUE JORGE NEWBERY, AR",2025-05-01,31,16.41
9,"MINISTRO PISTARINI, AR",2025-05-01,31,15.10


In [7]:
# Temperatura máxima del período por aeropuerto


conn.execute("""
    WITH maximas AS (
        SELECT aeropuerto, MAX(maxima_c) AS maxima_periodo_c
        FROM clima
        GROUP BY aeropuerto
    )
    SELECT
        c.aeropuerto,
        c.fecha,
        ROUND(c.maxima_c, 2) AS temperatura_maxima_c
    FROM clima AS c
    JOIN maximas AS m
        ON c.aeropuerto = m.aeropuerto
       AND c.maxima_c = m.maxima_periodo_c
    ORDER BY c.aeropuerto, c.fecha
""").df()

,aeropuerto,fecha,temperatura_maxima_c
0,"AEROPARQUE JORGE NEWBERY, AR",2025-02-11,36.28
1,"MINISTRO PISTARINI, AR",2025-02-03,39.00
2,"MINISTRO PISTARINI, AR",2025-02-10,39.00


In [8]:
# Días con máxima de al menos 30 °C y mínima de al menos 20 °C


conn.execute("""
    SELECT
        aeropuerto,
        COUNT(DISTINCT fecha) FILTER (
            WHERE maxima_c >= 30 AND minima_c >= 20
        ) AS dias_calidos
    FROM clima
    GROUP BY aeropuerto
    ORDER BY dias_calidos DESC, aeropuerto
""").df()

,aeropuerto,dias_calidos
0,"MINISTRO PISTARINI, AR",31
1,"AEROPARQUE JORGE NEWBERY, AR",26


In [9]:
# Comparación de la temperatura media y diferencia entre ambos aeropuertos

conn.execute("""
    SELECT
        a.fecha,
        ROUND(a.media_c, 2) AS aeroparque_c,
        ROUND(e.media_c, 2) AS ezeiza_c,
        ROUND(ABS(a.media_c - e.media_c), 2) AS diferencia_c
    FROM clima AS a
    JOIN clima AS e ON a.fecha = e.fecha
    WHERE a.aeropuerto = 'AEROPARQUE JORGE NEWBERY, AR'
      AND e.aeropuerto = 'MINISTRO PISTARINI, AR'
      AND a.media_c IS NOT NULL
      AND e.media_c IS NOT NULL
    ORDER BY ABS(a.media_c - e.media_c) DESC, a.fecha
    LIMIT 5
""").df()

,fecha,aeroparque_c,ezeiza_c,diferencia_c
0,2025-06-18,11.06,6.89,4.17
1,2025-05-19,13.44,9.33,4.11
2,2025-05-20,12.89,8.83,4.06
3,2025-06-26,9.94,6.33,3.61
4,2025-06-08,12.67,9.06,3.61


In [10]:
conn.close()

## Conclusiones

Se analizaron 472 registros diarios de 2025, con 236 observaciones de Aeroparque Jorge Newbery y 236 de Ezeiza Ministro Pistarini. Los datos abarcan de enero a agosto. No se encontraron fechas duplicadas por aeropuerto ni valores faltantes en las temperaturas media, máxima y mínima.

En base a lo que se pudo observar;

- Junio fue el mes con menor temperatura media en ambos aeropuertos: 11,09 °C en Aeroparque y 9,52 °C en Ezeiza.

- La temperatura máxima del período fue de 39 °C en Ezeiza y se registró dos veces, el 3 y el 10 de febrero. En Aeroparque, la máxima fue de 36,28 °C el 11 de febrero.

- Ezeiza tuvo 31 días con máxima de al menos 30 °C y mínima de al menos 20 °C. En Aeroparque hubo 26 días que cumplieron ambas condiciones.

- La mayor diferencia entre las temperaturas medias diarias se observó el 18 de junio. Ese día, Aeroparque registró 11,06 °C y Ezeiza 6,89 °C, una diferencia de 4,17 °C.